# SVG glyph path geometry viewer

Loads `References/dataset.zip`, lets you choose a class, and draws every SVG in that class with path commands color-coded.

The original glyph is shown faintly in the background. Path segments are overlaid by command type; segment end points are emphasized. All examples are normalized to the same visual bbox height.

In [ ]:
!pip -q install svgpathtools ipywidgets

import io, os, re, zipfile, requests
from pathlib import PurePosixPath
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath
from IPython.display import display, clear_output
import ipywidgets as widgets
from svgpathtools import parse_path, Line, CubicBezier, QuadraticBezier, Arc


In [ ]:
DATASET_URL = 'https://raw.githubusercontent.com/gasharva/svg-music/master/References/dataset.zip'

r = requests.get(DATASET_URL, timeout=60)
r.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(r.content))

svg_files = [n for n in zf.namelist() if n.lower().endswith('.svg')]
classes = sorted({PurePosixPath(n).parent.name for n in svg_files})
print(f'Loaded {len(svg_files)} SVG files in {len(classes)} classes')
print(classes[:20], '...' if len(classes) > 20 else '')


In [ ]:
COLORS = {
    'M': '#7f7f7f',
    'L': '#1f77b4',
    'H': '#17becf',
    'V': '#2ca02c',
    'C': '#ff7f0e',
    'S': '#ffbb78',
    'Q': '#9467bd',
    'T': '#c5b0d5',
    'A': '#d62728',
    'Z': '#8c564b',
}

SVG_NS = {'svg': 'http://www.w3.org/2000/svg'}
TOKEN_RE = re.compile(r'[AaCcHhLlMmQqSsTtVvZz]|[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?')
PARAMS = {'M':2,'L':2,'H':1,'V':1,'C':6,'S':4,'Q':4,'T':2,'A':7,'Z':0}

def parse_command_chunks(d):
    tokens = TOKEN_RE.findall(d or '')
    out = []
    i = 0
    cmd = None
    while i < len(tokens):
        if tokens[i].isalpha():
            cmd = tokens[i]
            i += 1
            if cmd.upper() == 'Z':
                out.append((cmd, []))
                continue
        if cmd is None:
            raise ValueError('Path data starts without a command')
        n = PARAMS[cmd.upper()]
        if i + n > len(tokens):
            break
        vals = list(map(float, tokens[i:i+n]))
        out.append((cmd, vals))
        i += n
        # extra coordinate pairs after M/m are implicit L/l
        if cmd.upper() == 'M':
            cmd = 'l' if cmd.islower() else 'L'
    return out

def command_segments(d):
    chunks = parse_command_chunks(d)
    cur = np.array([0.0, 0.0])
    sub_start = None
    prev_ctrl = None
    result = []

    def abspt(x, y, rel):
        p = np.array([x, y], dtype=float)
        return cur + p if rel else p

    for raw, v in chunks:
        c = raw.upper(); rel = raw.islower()
        start = cur.copy()
        if c == 'M':
            cur = abspt(v[0], v[1], rel); sub_start = cur.copy(); prev_ctrl = None
            result.append(('M', start, cur.copy(), None))
        elif c == 'L':
            end = abspt(v[0], v[1], rel); result.append(('L', start, end, Line(complex(*start), complex(*end)))); cur=end; prev_ctrl=None
        elif c == 'H':
            end = np.array([cur[0] + v[0] if rel else v[0], cur[1]]); result.append(('H', start, end, Line(complex(*start), complex(*end)))); cur=end; prev_ctrl=None
        elif c == 'V':
            end = np.array([cur[0], cur[1] + v[0] if rel else v[0]]); result.append(('V', start, end, Line(complex(*start), complex(*end)))); cur=end; prev_ctrl=None
        elif c == 'C':
            p1=abspt(v[0],v[1],rel); p2=abspt(v[2],v[3],rel); end=abspt(v[4],v[5],rel)
            seg=CubicBezier(complex(*start),complex(*p1),complex(*p2),complex(*end)); result.append(('C',start,end,seg)); cur=end; prev_ctrl=p2
        elif c == 'S':
            p1 = start if prev_ctrl is None else 2*start-prev_ctrl
            p2=abspt(v[0],v[1],rel); end=abspt(v[2],v[3],rel)
            seg=CubicBezier(complex(*start),complex(*p1),complex(*p2),complex(*end)); result.append(('S',start,end,seg)); cur=end; prev_ctrl=p2
        elif c == 'Q':
            p1=abspt(v[0],v[1],rel); end=abspt(v[2],v[3],rel)
            seg=QuadraticBezier(complex(*start),complex(*p1),complex(*end)); result.append(('Q',start,end,seg)); cur=end; prev_ctrl=p1
        elif c == 'T':
            p1 = start if prev_ctrl is None else 2*start-prev_ctrl
            end=abspt(v[0],v[1],rel)
            seg=QuadraticBezier(complex(*start),complex(*p1),complex(*end)); result.append(('T',start,end,seg)); cur=end; prev_ctrl=p1
        elif c == 'A':
            rx,ry,rot,large,sweep,x,y=v; end=abspt(x,y,rel)
            try: seg=Arc(complex(*start), complex(rx,ry), rot, bool(large), bool(sweep), complex(*end))
            except Exception: seg=Line(complex(*start), complex(*end))
            result.append(('A',start,end,seg)); cur=end; prev_ctrl=None
        elif c == 'Z' and sub_start is not None:
            end=sub_start.copy(); result.append(('Z',start,end,Line(complex(*start),complex(*end)))); cur=end; prev_ctrl=None
    return result

def sample_seg(seg, n=30):
    if seg is None: return np.empty((0,2))
    pts=[seg.point(t) for t in np.linspace(0,1,n)]
    return np.array([[p.real,p.imag] for p in pts])

def parse_svg(svg_bytes):
    root=ET.fromstring(svg_bytes)
    paths=[]
    for el in root.iter():
        if el.tag.split('}')[-1]=='path' and el.get('d'):
            paths.append(el.get('d'))
    return paths

def all_points(paths):
    pts=[]
    for d in paths:
        for _,_,end,seg in command_segments(d):
            if seg is not None: pts.extend(sample_seg(seg, 40))
            else: pts.append(end)
    return np.asarray(pts, dtype=float) if pts else np.zeros((0,2))


In [ ]:
def draw_example(ax, svg_bytes, title):
    paths = parse_svg(svg_bytes)
    pts = all_points(paths)
    if len(pts)==0:
        ax.set_title(title); ax.axis('off'); return

    xmin,ymin=pts.min(axis=0); xmax,ymax=pts.max(axis=0)
    h=max(ymax-ymin,1e-9); w=max(xmax-xmin,1e-9)
    # normalize bbox height to 1; preserve aspect ratio
    def norm(p): return np.column_stack(((p[:,0]-xmin)/h, (p[:,1]-ymin)/h))

    # faint background glyph
    for d in paths:
        for _,_,_,seg in command_segments(d):
            if seg is None: continue
            q=norm(sample_seg(seg,50))
            ax.plot(q[:,0],q[:,1],linewidth=5,alpha=.12)

    # command-colored overlay + fat end points
    for d in paths:
        for cmd,start,end,seg in command_segments(d):
            color=COLORS.get(cmd.upper(),'black')
            if cmd.upper()=='M':
                q=norm(np.array([end]))[0]
                ax.scatter([q[0]],[q[1]],s=46,color=color,zorder=5)
                continue
            if seg is not None:
                q=norm(sample_seg(seg,35))
                ax.plot(q[:,0],q[:,1],linewidth=2.2,color=color,zorder=3)
            qe=norm(np.array([end]))[0]
            ax.scatter([qe[0]],[qe[1]],s=34,color=color,edgecolors='white',linewidths=.5,zorder=6)

    ax.set_xlim(-0.05, w/h+0.05); ax.set_ylim(1.05,-0.05)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(title,fontsize=10)

def render_class(class_name):
    files=sorted([n for n in svg_files if PurePosixPath(n).parent.name==class_name])
    clear_output(wait=True)
    display(class_dropdown)
    legend='   '.join(f'{k}={v}' for k,v in COLORS.items())
    print(f'{class_name}: {len(files)} examples')
    print('Commands:', legend)
    if not files: return
    fig,axes=plt.subplots(1,len(files),figsize=(max(4,3.2*len(files)),4),squeeze=False)
    for ax,name in zip(axes[0],files):
        draw_example(ax,zf.read(name),PurePosixPath(name).name)
    plt.tight_layout(); plt.show()

class_dropdown=widgets.Dropdown(options=classes,description='Class:',layout=widgets.Layout(width='520px'))
class_dropdown.observe(lambda ch: render_class(ch['new']) if ch['name']=='value' and ch['type']=='change' else None,names='value')
render_class(class_dropdown.value)
